# 类与对象

学习目标：能用类组织对象的状态与操作，并正确选择属性和方法接口。

前置知识：变量赋值、对象引用与可变性、列表操作、条件判断、函数定义与参数传递。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 定义类并创建实例

一条学习笔记可以有标题和正文。class 语句定义一个类，调用类可以创建实例（instance）；下面的 StudyNote 是类名，note 是保存某个实例的变量。

类本身也是对象；“实例”强调一个对象与其所属类的关系，并不是与“对象”并列的另一种东西。

| 名称 | 中文名称／含义 | 本例中的对应项 |
| --- | --- | --- |
| class | 类；也指用于定义类的关键字 | StudyNote |
| instance | 实例；某个类的具体对象 | note 指向的对象 |
| attribute | 属性；按名称关联到对象的值 | title、text |

通过 note.title 这样的点号写法访问属性。这里的普通自定义实例允许在创建后添加数据属性；不能据此认为所有内置对象都允许添加任意属性。

In [1]:
class StudyNote:
    """保存一条学习笔记的标题和正文。"""


note = StudyNote()
note.title = "对象"
note.text = "同一个对象可以有多个引用。"

print(note.title)  # 预期：对象。
print(note.text)  # 预期：同一个对象可以有多个引用。
print(type(note) is StudyNote)  # 预期 True：该对象的类型是 StudyNote。

对象
同一个对象可以有多个引用。
True


## 2 self 与初始化

创建计数器时就应设置初始计数。\_\_init\_\_ 是初始化方法：对于本章的普通类，它会在新实例已经创建后、交给调用者前自动运行。实际创建实例由 \_\_new\_\_ 负责；定制创建过程留到第 31 章“描述器与元类”。

方法定义中的第一个参数通常叫 self，表示当前实例。self 不是关键字，但应遵循这个命名约定。self.count 保存实例状态；start 和 amount 则是方法调用中的局部参数，本例分别表示初始计数和增加的次数，均使用非负整数。

Counter(3) 把 3 传给初始化方法的 start，self 由 Python 提供。\_\_init\_\_ 应返回 None，通常省略 return；不要在其中返回新实例。普通方法 add 则可以返回运算结果。

In [2]:
class Counter:
    """保存一个计数，并支持增加指定次数。"""

    def __init__(self, start=0):
        """用给定计数初始化当前实例。"""
        self.count = start

    def add(self, amount=1):
        """增加计数并返回更新后的值。"""
        self.count += amount
        return self.count


counter = Counter(3)
print(counter.count)  # 预期 3：初始化已经完成。
print(counter.add(2))  # 预期 5：本次调用中的 self 就是 counter 所指实例。

3
5


两个实例可以各自保存 count。给变量赋值却不会创建新实例：别名仍然指向原对象，这与列表的引用规则相同。

下面沿用 Counter 的定义，重新创建两个计数器，比较“再次调用类”和“给实例起别名”的差别。

In [3]:
left = Counter()
right = Counter()
alias = left
alias.add()

print(left.count, right.count)  # 预期 1 0：只更新了 left 对应的实例。
print(alias is left)  # 预期 True：赋值没有复制计数器。
print(left is right)  # 预期 False：两次实例化得到两个实例。

1 0
True
False


## 3 实例方法与绑定

在类中定义的普通函数，通过实例访问时会形成绑定方法（bound method），把实例与函数关联起来。调用时，绑定的实例会自动补到参数列表的开头。

对于 Counter.add，left.add(2) 相当于 Counter.add(left, 2)；left 表示 Counter 实例，2 是增加的次数。通过实例调用时，不再手动传 self。

方法也使用属性名访问，因此属于广义的属性；“数据属性与方法”按用途区分，“类属性与实例属性”则关注名称保存的位置，两组分类会有交叉。

In [4]:
left = Counter(10)
right = Counter(10)

print(left.add(2))  # 预期 12：自动传入 left。
print(Counter.add(right, 2))  # 预期 12：经类访问函数时显式传入实例。

12
12


绑定方法可以保存后再调用。它记住的是实例对象，而不是后来可能改指向的变量名。

下面仍使用 Counter。方法的 \_\_self\_\_ 属性指向绑定的实例，\_\_func\_\_ 属性指向原函数；这里只用它们确认绑定关系。

In [5]:
counter = Counter()
original = counter
step = counter.add
counter = Counter(50)
step(3)

print(original.count, counter.count)  # 预期 3 50：step 仍操作原实例。
print(step.__self__ is original)  # 预期 True：绑定对象没有随变量改指向。
print(step.__func__ is Counter.add)  # 预期 True：底层函数是 Counter.add。

3 50
True
True


## 4 类属性与实例属性

类体中直接赋值的普通属性保存在类上；通过 self 赋值的数据属性保存在实例上。下面用类属性 theme 表示笔记显示的默认主题，用实例属性 name 区分各份笔记。

| 名称 | 中文名称／含义 | 本例中的读写入口 |
| --- | --- | --- |
| class attribute | 类属性；保存在类上的属性 | NoteStyle.theme |
| instance attribute | 实例属性；保存在某个实例上的属性 | first.name |

对于这里的普通数据属性，实例没有同名属性时会读取类属性。此处先讨论没有特殊访问逻辑的属性；后面的 property 会接管指定属性的读写。

In [6]:
class NoteStyle:
    """保存笔记名称，并提供默认显示主题。"""

    theme = "浅色"

    def __init__(self, name):
        """为当前笔记保存名称。"""
        self.name = name


first = NoteStyle("语法")
second = NoteStyle("对象")
print(NoteStyle.theme)  # 预期：浅色。
print(first.name, first.theme)  # 预期：语法 浅色。
print(second.name, second.theme)  # 预期：对象 浅色。

浅色
语法 浅色
对象 浅色


沿用 NoteStyle。给 first.theme 赋值会建立同名实例属性，遮蔽类上的默认值，不会修改 NoteStyle.theme。

修改类属性会影响仍然从类上读取该值的实例。删除这里的同名实例属性后，读取会再次找到类上的值。

In [7]:
NoteStyle.theme = "浅色"
first = NoteStyle("语法")
second = NoteStyle("对象")
first.theme = "深色"
NoteStyle.theme = "护眼"

print(first.theme, second.theme, NoteStyle.theme)
# 预期：深色 护眼 护眼。first 的主题保存在实例上。

del first.theme
print(first.theme)  # 预期：护眼。删除覆盖值后，重新使用类上的主题。

深色 护眼 护眼
护眼


## 5 可变状态的归属

每份阅读清单应有自己的书目，类体中的列表却由实例共享。下面保留这个反例：first、second 各有 owner，但没有各自的 titles，读取时都找到类属性中的同一个列表。

![可变类属性：两个实例找到同一个列表](image/illustration/09-01-shared-class-state.svg)

图示：共享类属性的反例。图中两个实例都没有自己的 titles，因而读取同一类属性列表。

self.titles.append(title) 修改找到的列表，没有为 self.titles 重新赋值，因此不会自动创建实例列表。title 是本次添加的书名。先核对 second.titles 与 is 的结果，再比较后面在初始化中创建列表的修正版。

In [8]:
class BadReadingList:
    """演示把各人书目错误地保存在共享类属性中的后果。"""

    titles = []

    def __init__(self, owner):
        """保存阅读清单的持有人。"""
        self.owner = owner

    def add(self, title):
        """向当前找到的列表添加书名。"""
        self.titles.append(title)


first = BadReadingList("小林")
second = BadReadingList("小周")
first.add("Python 教程")

print(second.titles)  # 预期 ['Python 教程']：小周的清单被意外影响。
print(first.titles is second.titles)  # 预期 True：读取的是同一个列表。

['Python 教程']
True


把空列表的创建放进 \_\_init\_\_，每次初始化都会得到一个新的列表，并把它保存在当前实例的 titles 属性中。

这里的独立性来自“每次新建列表”。不需要在每次 add 调用中重新建表，否则之前的书目也无法保留。

In [9]:
class ReadingList:
    """为每位持有人保存独立的阅读清单。"""

    def __init__(self, owner):
        """保存持有人，并为当前实例新建书目列表。"""
        self.owner = owner
        self.titles = []

    def add(self, title):
        """向当前实例的清单添加书名。"""
        self.titles.append(title)


first = ReadingList("小林")
second = ReadingList("小周")
first.add("Python 教程")

print(first.titles, second.titles)  # 预期：['Python 教程'] []。
print(first.titles is second.titles)  # 预期 False：列表分别创建。

['Python 教程'] []
False


实例属性也可能引用同一个可变对象。下面沿用 ReadingList，主动让两个实例的 titles 指向同一列表，观察共享如何重新出现。

因此，“属性属于哪个实例”和“属性引用的对象是否共享”是两个问题；写成 self.titles 并不保证之后永远独立。

In [10]:
first = ReadingList("小林")
second = ReadingList("小周")
second.titles = first.titles
first.add("函数")

print(first is second)  # 预期 False：清单实例仍是两个。
print(first.titles is second.titles)  # 预期 True：实例属性引用了同一列表。
print(second.titles)  # 预期 ['函数']：共享引用导致内容同步变化。

False
True
['函数']


## 6 封装与名称约定

封装把状态及其相关操作组织在一起，并为使用者提供清楚的接口。下面的书架用 add 添加书名，用 list\_titles 获取供查看的列表，内部列表命名为 \_titles。

单个前导下划线表示“非公开，实现细节可能变化”的约定，并不禁止外部访问。使用者应优先调用公开接口；本例返回列表的浅拷贝，避免调用者增删返回值时直接改动书架。

本例的列表元素约定为书名字符串。浅拷贝只复制外层列表；如果以后改成嵌套可变记录，还需另外考虑内部元素的共享。

In [11]:
class BookShelf:
    """保存书名，并通过公开方法提供添加和查看操作。"""

    def __init__(self):
        """创建当前书架的内部列表。"""
        self._titles = []

    def add(self, title):
        """向书架添加一个书名字符串。"""
        self._titles.append(title)

    def list_titles(self):
        """返回书名列表的浅拷贝。"""
        return self._titles.copy()


shelf = BookShelf()
shelf.add("Python 教程")
view = shelf.list_titles()
view.append("临时条目")

print(view)  # 预期：['Python 教程', '临时条目']。
print(shelf.list_titles())  # 预期 ['Python 教程']：书架没有新增临时条目。
print(shelf._titles)  # 仍可读到 ['Python 教程']；这里只为观察命名约定。

['Python 教程', '临时条目']
['Python 教程']
['Python 教程']


名称改写（name mangling）用于减少意外的同名冲突，尤其是继承扩展中的冲突；继承在第 10 章展开。类定义中以至少两个下划线开头、至多一个下划线结尾的名称会按所在类名改写。

例如 ClickCounter 中的 \_\_count 会改写为 \_ClickCounter\_\_count。这个名称仍可访问，因此名称改写不是访问控制，不应把它当作保密机制。

| 名称 | 中文名称／含义 | 本章用法 |
| --- | --- | --- |
| \_titles | 单前导下划线的非公开名称 | 约定使用者通过公开接口操作 |
| \_\_count | 会触发名称改写的名称 | 观察改写后的实际名称 |
| \_\_init\_\_ | 双下划线开头和结尾的特殊方法名 | 初始化；不属于上述名称改写形式 |

In [12]:
class ClickCounter:
    """通过计数观察双前导下划线的名称改写。"""

    def __init__(self):
        """初始化内部计数。"""
        self.__count = 0

    def increment(self):
        """增加一次点击。"""
        self.__count += 1

    def value(self):
        """返回当前内部计数。"""
        return self.__count


clicks = ClickCounter()
clicks.increment()
print(clicks.value())  # 预期 1：方法使用的是改写后的名称。
print(clicks._ClickCounter__count)  # 预期 1：外部仍可访问，不是权限限制。

clicks.__count = 99  # 在类外使用此名称，给实例添加的是另一个属性。
clicks.increment()
print(clicks.__count, clicks.value())  # 预期 99 2：两个名称没有指向同一属性。

1
1
99 2


## 7 用 property 管理属性读写

### 7.1 用 getter 计算只读属性

若读取属性时需要计算，property（特征属性）可以保留点号访问的写法，并自动运行取值函数 getter。读取 queue.count 不加调用括号；queue 表示下面的阅读队列实例。

@property 写在函数定义之前，使用内置装饰器把该函数变成同名特征属性。后面的 @classmethod、@staticmethod 也是内置装饰器；本章学习直接用法，一般机制在第 14 章“装饰器”展开。

property 定义在类上，但这里管理的是每个实例的属性访问；“定义在类上”与“管理实例数据”并不冲突。只提供 getter 时该属性只读；若只需普通存取，没有必要把每个数据属性都改成 property。

In [13]:
class ReadingQueue:
    """保存待阅读书目，并在读取时计算书目数量。"""

    def __init__(self):
        """为当前队列新建书目列表。"""
        self._titles = []

    def add(self, title):
        """向队列添加书名。"""
        self._titles.append(title)

    @property
    def count(self):
        """返回当前书目数量，不另存一份可能过期的计数。"""
        return len(self._titles)


queue = ReadingQueue()
print(queue.count)  # 预期 0：访问时运行 getter。
queue.add("对象")
queue.add("函数")
print(queue.count)  # 预期 2：根据当前列表重新计算，没有缓存旧值。

0


2


只读意味着不能通过这个属性直接赋值，不意味着整个实例不能变化。例如 ReadingQueue 仍可通过 add 改变书目。

给没有 setter 的 count 赋值会引发 AttributeError，表示这次属性赋值失败。下面先借用 try/except AttributeError，只处理这个预期异常，使 Notebook 可以继续执行；完整异常处理在第 11 章讲解。

In [14]:
queue = ReadingQueue()
queue.add("类")

In [15]:
# 预期 AttributeError：直接观察原始异常，之后继续运行下一单元。
queue.count = 10

AttributeError: property 'count' of 'ReadingQueue' object has no setter

In [16]:
print(queue.count)  # 预期 1：仍由书目列表决定，没有被写成 10。

1


### 7.2 用 setter 校验新值

赋值函数 setter 可以在写入前检查数据。下面的 Volume 用 level 表示 0 到 100（含端点）的音量百分比；value 表示准备写入的新数值，示例输入使用 int 或 float。

@level.setter 给已定义的 level 增加设置函数，设置函数仍命名为 level。读取和赋值分别通过 getter 和 setter，实际值保存在 \_level 中；在 setter 里写 self.level = value 会再次触发自己，所以应写 self.\_level。

raise ValueError 会主动拒绝范围不合适的数值。先校验再保存，失败时不会改动旧值。\_\_init\_\_ 也通过 self.level 赋值，从而复用相同规则。

In [17]:
class Volume:
    """通过 level 属性管理范围为 0 到 100 的音量百分比。"""

    def __init__(self, level=50):
        """经由公开属性校验并设置初始音量。"""
        # 初始化也走公开 setter，创建和后续赋值使用同一条范围规则。
        self.level = level

    @property
    def level(self):
        """返回当前音量百分比。"""
        return self._level

    @level.setter
    def level(self, value):
        """保存有效音量；越界时引发 ValueError。"""
        if not 0 <= value <= 100:
            raise ValueError("音量必须在 0 到 100 之间")
        self._level = value


volume = Volume(20)
print(volume.level)  # 预期 20：初始化时也经过 setter。
for level in [0, 100]:
    volume.level = level
    print(volume.level)  # 依次预期 0、100：两个端点都允许。

20
0
100


沿用 Volume，分别观察更新失败和初始化失败。这里用 except ValueError 处理已知的越界输入；as error 将本次异常交给变量 error，便于打印具体原因。

property 的约束依赖调用者通过 level 操作。单下划线的 \_level 仍能被外部直接修改，所以这是一种接口设计，不是权限边界。

In [18]:
volume = Volume(30)

In [19]:
# 预期 ValueError：直接观察原始异常，之后继续运行下一单元。
# 预期：音量必须在 0 到 100 之间。
volume.level = -1

ValueError: 音量必须在 0 到 100 之间

In [20]:
print(volume.level)  # 30：拒绝后保留原值。

30


In [21]:
# 预期 ValueError：直接观察原始异常，之后继续运行下一单元。
Volume(101)

ValueError: 音量必须在 0 到 100 之间

## 8 classmethod 与 staticmethod

### 8.1 用 classmethod 提供替代构造入口

@classmethod 定义类方法，自动接收的第一个参数是类，按惯例命名为 cls。它不需要先有一个实例，适合提供另一种输入形式的构造入口。

下面的 Duration 保存秒数，seconds 表示秒数，minutes 表示分钟数，本例均约定为非负整数。from\_minutes 将分钟换算为秒，再调用 cls 创建实例。

这个替代构造方法返回新实例；它最终仍通过普通实例化调用 \_\_init\_\_ 完成初始化，并没有让 \_\_init\_\_ 承担实例创建的职责。

In [22]:
class Duration:
    """保存一段时长，并支持按分钟构造。"""

    def __init__(self, seconds):
        """保存秒数，构造时拒绝负值。"""
        if seconds < 0:
            raise ValueError("时长不能为负数")
        self.seconds = seconds

    @classmethod
    def from_minutes(cls, minutes):
        """按分钟换算秒数，并创建一个当前类的实例。"""
        return cls(minutes * 60)


short_break = Duration(30)
long_break = Duration.from_minutes(2)
print(short_break.seconds, long_break.seconds)  # 预期 30 120：两种输入入口。
print(type(long_break) is Duration)  # 预期 True：类方法返回新建的实例。

30 120
True


类方法也能经由实例调用，但自动传入的仍是所属类，不是该实例。下面沿用 Duration，观察经实例调用的 from\_minutes 不会读取或更新该实例的 seconds。

同样可以用绑定方法的 \_\_self\_\_ 查看接收对象；在类方法上，它指向类。日常使用替代构造入口时，写成 Duration.from\_minutes 更能表明意图。

In [23]:
duration = Duration(90)
new_duration = duration.from_minutes(3)

print(duration.seconds, new_duration.seconds)  # 预期 90 180：原时长没有修改。
print(new_duration is duration)  # 预期 False：得到另一个实例。
print(Duration.from_minutes.__self__ is Duration)  # 预期 True：绑定的是类。

90 180
False
True


### 8.2 用 staticmethod 提供相关工具

@staticmethod 定义静态方法，不自动传入 self 或 cls。它的所有输入都要显式提供；既可以通过类调用，也可以通过实例调用。

下面的 TextBlock 保存正文。count\_parts 接收字符串 text，按空白分隔统计片段数；这只是分隔计数，不是中文分词。part\_count 是 property，用同一工具计算当前实例的正文。

如果只需对字符串计数，普通模块函数 count\_parts(text) 就足够。本例把工具放在 TextBlock 中，是因为它与已存在的正文状态和读取操作相关，不必为单个工具函数另外创建一个类。

| 名称 | 中文名称／含义 | 自动接收的对象 |
| --- | --- | --- |
| instance method | 实例方法；操作某个实例 | 实例，惯例参数名为 self |
| classmethod | 类方法；如替代构造入口 | 类，惯例参数名为 cls |
| staticmethod | 静态方法；类中的相关工具 | 无 |
| property | 特征属性；管理点号读取或赋值 | 本章 getter、setter 接收实例 |

In [24]:
class TextBlock:
    """保存正文，并提供按空白分隔的片段计数。"""

    def __init__(self, text):
        """保存正文字符串。"""
        self.text = text

    # 计数只需要传入的文本，不依赖实例状态。
    @staticmethod
    def count_parts(text):
        """统计空白分隔的片段数，空文本计为零。"""
        return len(text.split())

    @property
    def part_count(self):
        """返回当前正文的片段数。"""
        return self.count_parts(self.text)


block = TextBlock("read small examples")
print(TextBlock.count_parts("read code"))  # 预期 2：不必创建实例。
print(block.count_parts("read code"))  # 预期 2：不会自动传入 block。
print(block.part_count)  # 预期 3：property 读取的是当前实例的正文。
block.text = "read small examples daily"
print(block.part_count)  # 预期 4：正文变化后重新计算。
print(TextBlock.count_parts("  "))  # 预期 0：纯空白没有片段。

2
2
3
4
0


## 本章小结

（1）类也是对象；调用普通类得到实例，\_\_init\_\_ 初始化已创建的实例。self 是实例参数的惯用名称。

（2）绑定方法保存实例与原函数的关联。经类访问普通函数时，要显式传入实例；类方法自动接收类，静态方法没有隐式参数。

（3）类属性可以提供共享默认值，实例属性保存各自状态。可变对象是否共享，还取决于属性实际引用了哪个对象。

（4）公开方法与 property 可以组织读写规则；单下划线表达非公开约定，名称改写减少冲突，两者都不是访问控制。

自查：能否说明一个值保存在哪里、一次调用自动传入什么对象，以及一次修改会影响哪些实例？

## 练习

### 练习 1：预测属性查找与状态变化

先写下下面两次 print 的输出，再说明每个值来自实例还是类、两个 items 是否共享。运行后逐项核对实际输出；不要只根据变量名称猜测。

In [25]:
class LessonBoard:
    """保存课程看板条目，并提供默认级别。"""

    level = "入门"

    def __init__(self):
        """为每个看板新建条目列表。"""
        self.items = []


first = LessonBoard()
second = LessonBoard()
first.level = "进阶"
first.items.append("方法")
LessonBoard.level = "基础"
# 先预测主题与条目列表，再分别核对类属性覆盖和实例列表。
print(first.level, second.level)
print(first.items, second.items)

进阶 基础
['方法'] []


### 练习 2：实现独立且便于使用的清单

实现 TaskList：每个实例在初始化时创建非公开列表 \_tasks，add 接收任务字符串，list\_tasks 返回列表的浅拷贝。

检查两件事：只给一个实例添加任务时，另一个实例仍为空；向 list\_tasks 的返回值追加字符串时，原实例的任务不变。解释为什么“每个实例新建列表”和“返回时复制列表”解决的是不同位置的共享问题。

In [26]:
# 在此实现 TaskList，并用两个实例及一个返回列表核对题目要求。
pass

### 练习 3：组合属性校验与替代构造入口

实现 PracticeVolume，支持可读写的 level，范围为 0 到 100（含端点）。初始化和后续赋值都应经过相同校验，越界时引发 ValueError，并保留已有实例的旧值。

增加类方法 from\_fraction：参数 fraction 是 0 到 1 的音量比例，换算为百分比后创建实例。再增加静态方法 is\_valid\_level：参数 value 是待检查的数值，返回它是否落在有效范围内；让 setter 复用该判断。

检查 from\_fraction(0.6) 得到的 level 为 60.0，0 与 100 可以赋值，写入 101 会失败且不改旧值。输入在本题中约定为普通 int 或 float 数值；思考哪些调用需要已有实例，哪些可以直接通过类完成。

In [27]:
# 在此实现 PracticeVolume；只捕获检查中预期的 ValueError。
# 自行加入有效边界、无效赋值和替代构造入口的核对代码。
pass

## 参考与引用来源

| 网站 | 已核查的版本、位置与支持内容 |
| --- | --- |
| Python 官方文档（docs.python.org） | Python 3.12：[属性术语](https://docs.python.org/zh-cn/3.12/glossary.html#term-attribute)、[类术语](https://docs.python.org/zh-cn/3.12/glossary.html#term-class)；教程第 9 节的[类定义与实例](https://docs.python.org/zh-cn/3.12/tutorial/classes.html#a-first-look-at-classes)、[方法对象](https://docs.python.org/zh-cn/3.12/tutorial/classes.html#method-objects)、[类和实例变量](https://docs.python.org/zh-cn/3.12/tutorial/classes.html#class-and-instance-variables)、[普通属性查找与 self 约定](https://docs.python.org/zh-cn/3.12/tutorial/classes.html#random-remarks)、[非公开约定与名称改写](https://docs.python.org/zh-cn/3.12/tutorial/classes.html#private-variables)；语言参考的[实例初始化](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__init__)、[实例创建](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__new__)、[绑定方法的属性](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#instance-methods)、[装饰器定义规则](https://docs.python.org/zh-cn/3.12/reference/compound_stmts.html#function-definitions)；内置函数文档的[property](https://docs.python.org/zh-cn/3.12/library/functions.html#property)、[classmethod](https://docs.python.org/zh-cn/3.12/library/functions.html#classmethod)、[staticmethod](https://docs.python.org/zh-cn/3.12/library/functions.html#staticmethod)；[可变序列的 append 与 copy](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#typesseq-mutable)、[str.split 的空白分隔规则](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#str.split)；[处理选定异常](https://docs.python.org/zh-cn/3.12/tutorial/errors.html#handling-exceptions)、[主动引发异常](https://docs.python.org/zh-cn/3.12/tutorial/errors.html#raising-exceptions)、[AttributeError](https://docs.python.org/zh-cn/3.12/library/exceptions.html#AttributeError)、[ValueError](https://docs.python.org/zh-cn/3.12/library/exceptions.html#ValueError)，支持只读属性和越界赋值的反例。 |